In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from kooplearn._linalg import eigh_rank_reveal, spd_neg_pow, weighted_norm
from kooplearn.datasets import compute_prinz_potential_eig, make_prinz_potential
from kooplearn.kernel import KernelRidge


def operator_norm_error(true_operator: np.ndarray, estimated_operator: np.ndarray):
    r"""Operator norm error proxy for a Koopman estimator.

    Computes the operator norm discrepancy between the true action
    :math:`A_\pi S` and the estimated action :math:`S \widehat{G}`:

    .. math::

        \mathcal{E}(\widehat{G}) := \|A_\pi S - S \widehat{G}\|.

    Since "kooplearn" does not currently expose :math:`A_\pi` or the embedding
    operator :math:`S` explicitly, this function works with their actions on a
    common finite-dimensional representation. In practice, the caller should pass
    matrices or vectors representing the two quantities to be compared.
    """
    true_operator = np.asanyarray(true_operator)
    estimated_operator = np.asanyarray(estimated_operator)

    if true_operator.shape != estimated_operator.shape:
        raise ValueError(
            "true_operator and estimated_operator must have the same "
            f"shape, got {true_operator.shape} and "
            f"{estimated_operator.shape}."
        )

    diff = true_operator - estimated_operator
    if diff.ndim == 1:
        return float(np.linalg.norm(diff))
    return float(np.linalg.norm(diff, ord=2))


def metric_distortion(psi, C):
    r"""Empirical metric distortion :math:`\widehat\eta_i = \|\widehat\psi_i\|_{\mathcal H} /
    \sqrt{\langle \widehat C \widehat\psi_i, \widehat\psi_i\rangle}`.

    Parameters
    ----------
    psi : ndarray, shape (n,) or (n, k)
        Eigenfunction(s) evaluated at the *training* points. If 2D, each
        column is treated as a separate eigenfunction (see `weighted_norm`).
    C : ndarray, shape (n, n)
        Empirical (kernel-based) covariance, i.e. ``model.kernel_X / n_samples``.
    """
    psi = np.asarray(psi)
    n = C.shape[0]

    # ||psi||_H via the reproducing property: needs the *inverse* Gram, since
    # C = K_X / n is the Gram-based covariance, not the RKHS metric itself.
    C_inv = spd_neg_pow(C * n, exponent=-1.0)  # i.e. K_X^{-1}
    rkhs_norm = weighted_norm(psi, M=C_inv)

    # <C psi, psi> = (1/n)||psi(X)||_2^2, i.e. weighted_norm with M=None, squared, over n
    empirical_norm = weighted_norm(psi, M=None) / np.sqrt(n)

    with np.errstate(divide="ignore", invalid="ignore"):
        eta = rkhs_norm / empirical_norm
    eta = np.where(empirical_norm > 0, eta, np.nan)
    return eta if psi.ndim == 2 else float(eta)


def spectral_bias(eigenfunction, C, rho):
    r"""Empirical spectral bias :math:`\hat s_i = \widehat\eta_i \, \rho_{r+1}`."""
    eta = metric_distortion(eigenfunction, C)
    s_hat = eta * rho
    return float(s_hat), eta


def _top_sv(C, r):
    """(r+1)-st eigenvalue of a symmetric PSD matrix, via eigh_rank_reveal."""
    raw_vals, raw_vecs = np.linalg.eigh(np.asarray(C))
    _, top_vals, _ = eigh_rank_reveal(raw_vals, raw_vecs, rank=r + 1)
    if len(top_vals) <= r:
        return 0.0
    return float(top_vals[-1])


# --- truncation helpers ---
def pcr_truncation(C, r):
    r""":math:`\rho_{r+1}(\widehat G^{PCR}) = \sigma_{r+1}(\widehat C)`."""
    return _top_sv(C, r)


# kDMD uses the same (r+1)-st eigenvalue of the empirical covariance as PCR
kdmd_truncation = pcr_truncation


def rrr_truncation(C, T, r, cutoff=None):
    r""":math:`\rho_{r+1}(\widehat G^{RRR}) = \sigma_{r+1}(\widehat C^{-1/2}\widehat T)`."""
    C_inv_sqrt = spd_neg_pow(np.asarray(C), exponent=-0.5, cutoff=cutoff)
    A = C_inv_sqrt @ np.asarray(T)
    svals = np.linalg.svd(A, compute_uv=False)
    if r >= len(svals):
        return 0.0
    return float(svals[r])


# --- spectral gap (top-two magnitude eigenvalues) ---
def spectral_gap(eigenvalues):
    mags = np.sort(np.abs(eigenvalues))[::-1]
    return float(mags[0] - mags[1]) if len(mags) > 1 else np.nan


# --- spurious eigenvalues vs reference ---


def spurious_ref(est, ref, delta):
    dist = np.abs(est[:, None] - ref[None, :])
    return int(np.sum(dist.min(axis=1) > delta))


def spurious_residual(eigenvalues, psi_X_val, psi_Y_val, delta, relative=True):
    r"""Data-driven spurious-eigenpair check (see paper Appendix C, Remark 4).

    Flags eigenpairs that fail the empirical consistency check
    :math:`\hat\psi_i(y_j) \approx \hat\lambda_i \hat\psi_i(x_j)` on a
    held-out validation set, i.e. large :math:`\|(\widehat Z - \widehat S\widehat G)\hat\psi_i\|`.

    Parameters
    ----------
    eigenvalues : ndarray, shape (r,)
        Estimated eigenvalues, same order as columns of psi_X_val/psi_Y_val.
    psi_X_val : ndarray, shape (n_val, r)
        Eigenfunctions evaluated at validation inputs x_j.
    psi_Y_val : ndarray, shape (n_val, r)
        Same eigenfunctions evaluated at the corresponding outputs y_j = f(x_j).
    delta : float
        Threshold on the (optionally normalized) residual.
    relative : bool
        If True, normalize residual by ||psi_X_val||, giving a scale-free score.

    Returns
    -------
    n_spurious : int
    scores : ndarray, shape (r,)
    """
    eigenvalues = np.asarray(eigenvalues)
    n_val = psi_X_val.shape[0]

    resid = psi_Y_val - psi_X_val * eigenvalues[None, :]
    resid_norm = weighted_norm(resid) / np.sqrt(n_val)

    if relative:
        base_norm = weighted_norm(psi_X_val) / np.sqrt(n_val)
        scores = np.where(base_norm > 0, resid_norm / base_norm, np.nan)
    else:
        scores = resid_norm

    n_spurious = int(np.sum(scores > delta))
    return n_spurious, scores


# --- compilation function for analysing spectral metrics ---
def analyse_spectrum(modes_records, trials_records, out_prefix):
    modes_df = pd.DataFrame(modes_records).copy()
    trials_df = pd.DataFrame(trials_records).copy()

    summary = modes_df.groupby(
        ["kernel", "kind", "method", "eigenfunction_id"], as_index=False
    ).agg(
        n=("spectral_bias", "size"),
        bias_mean=("spectral_bias", "mean"),
        bias_std=("spectral_bias", "std"),
        dist_mean=("metric_distortion", "mean"),
        trunc_mean=("truncation", "mean"),
        spurious_mean=("residual_spurious_score", "mean"),
        spurious_std=("residual_spurious_score", "std"),
    )
    # modes_df = modes_df.merge(
    #    trials_df[["trial", "spectral_gap"]].rename(columns={"spectral_gap": "gap"}),
    #    on="trial",
    #    how="left",
    # )
    corr_df = (
        modes_df.groupby(["kernel", "kind", "method"])
        .apply(lambda g: g["spectral_bias"].corr(g["gap"]))
        .rename("bias_gap_corr")
        .reset_index()
    )

    summary.to_csv(f"{out_prefix}_summary.csv", index=False)
    modes_df.to_csv(f"{out_prefix}_metrics.csv", index=False)
    trials_df.to_csv(f"{out_prefix}_trials.csv", index=False)
    corr_df.to_csv(f"{out_prefix}_corr.csv", index=False)

    fig1, ax = plt.subplots(figsize=(6.5, 4.5))
    for (kernel, kind, method), g in modes_df.groupby(["kernel", "kind", "method"]):
        ax.scatter(
            g["spectral_bias"],
            g["gap"],
            s=20,
            alpha=0.7,
            label=f"{kernel}, {kind} / {method}",
        )
    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Spectral gap")
    ax.legend(frameon=False, fontsize=8)
    ax.set_title("Spectral bias vs Spectral gap")
    fig1.tight_layout()
    fig1.savefig(f"{out_prefix}_gap_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig1)

    fig2, ax = plt.subplots(figsize=(6.5, 4.5))
    for (kernel, kind, method), g in modes_df.groupby(["kernel", "kind", "method"]):
        ax.scatter(
            g["spectral_bias"],
            g["residual_spurious_score"],
            s=20,
            alpha=0.7,
            label=f"{kernel}, {kind} / {method}",
        )
    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Residual spurious score")
    ax.legend(frameon=False, fontsize=8)
    ax.set_title("Spectral bias vs Residual spurious score")
    fig2.tight_layout()
    fig2.savefig(f"{out_prefix}_spurious_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig2)

    return modes_df, trials_df, summary, corr_df, fig1, fig2


In [ ]:
from itertools import product

# ============================================================
# 1. Data
# ============================================================

x = np.linspace(-2, 2, 2048 + 1)
gamma = 1.0
sigma = 2.0
dt = 1e-4
subsample = 100
n_train = 2000
n_val = 500
n_trials = 10
n_show = 3

n_steps_train = n_train * subsample
n_steps_val = n_val * subsample

data = make_prinz_potential(
    X0=0,
    n_steps=n_steps_train,
    gamma=gamma,
    sigma=sigma,
    random_state=0,
)
data = data.iloc[::subsample][:n_train]

data_val = make_prinz_potential(
    X0=0,
    n_steps=n_steps_val,
    gamma=gamma,
    sigma=sigma,
    random_state=999,
)
data_val = data_val.iloc[::subsample][:n_val]

X_val = data_val.iloc[:-1].values
Y_val = data_val.iloc[1:].values

vals_ref = compute_prinz_potential_eig(gamma, sigma, dt, num_components=5)


# ============================================================
# 2. Custom kernels from the attachment
# ============================================================


def anisotropic_rbf_kernel(X, Y=None, Lambda=None):
    X = np.asarray(X)
    Y = X if Y is None else np.asarray(Y)

    if X.ndim == 1:
        X = X[:, None]
    if Y.ndim == 1:
        Y = Y[:, None]

    if Lambda is None:
        Lambda = np.eye(X.shape[1])

    Lambda = np.asarray(Lambda)
    Lambda_inv = np.linalg.inv(Lambda)

    K = np.empty((X.shape[0], Y.shape[0]))
    for i, x0 in enumerate(X):
        diff = Y - x0
        K[i] = np.exp(-np.sum((diff @ Lambda_inv) * diff, axis=1))
    return K


# ============================================================
# Spectral kernels using compute_prinz_potential_eig
# ============================================================


def prinz_feature_map(X, gamma, sigma, dt, m=5):
    X = np.asarray(X)
    if X.ndim == 1:
        X = X[:, None]

    mu, Phi = compute_prinz_potential_eig(
        gamma=gamma,
        sigma=sigma,
        dt=dt,
        eval_right_on=X,
        num_components=m,
    )

    Phi = np.asarray(Phi, dtype=float)
    norms = np.linalg.norm(Phi, axis=0, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    Phi = Phi / norms
    return np.asarray(mu, dtype=float), Phi

def spectral_matched_kernel(gamma, sigma, dt, m=5, weights=None):
    def kernel(X, Y=None):
        mu_X, Phi_X = prinz_feature_map(X, gamma=gamma, sigma=sigma, dt=dt, m=m)
        if Y is None:
            mu_Y, Phi_Y = mu_X, Phi_X
        else:
            mu_Y, Phi_Y = prinz_feature_map(Y, gamma=gamma, sigma=sigma, dt=dt, m=m)

        w = np.abs(mu_X[:m]) if weights is None else np.asarray(weights, dtype=float)[:m]
        return (Phi_X[:, :m] * w) @ Phi_Y[:, :m].T

    kernel.__name__ = "spectral_matched"
    return kernel


def spectral_mismatched_kernel(gamma, sigma, dt, m=5, nu=1.0, perm=None):
    def kernel(X, Y=None):
        mu_X, Phi_X = prinz_feature_map(X, gamma=gamma, sigma=sigma, dt=dt, m=m)
        if Y is None:
            mu_Y, Phi_Y = mu_X, Phi_X
        else:
            mu_Y, Phi_Y = prinz_feature_map(Y, gamma=gamma, sigma=sigma, dt=dt, m=m)

        mu = np.abs(mu_X[:m])
        p = np.arange(len(mu))[::-1] if perm is None else np.asarray(perm)
        w = mu[p][:m] ** (2 * nu)
        return (Phi_X[:, :m] * w) @ Phi_Y[:, :m].T

    kernel.__name__ = "spectral_mismatched"
    return kernel

# ============================================================
# 3. Kernel panel
# ============================================================

Lambda_1d = np.array([[0.5]])


def anisotropic_kernel(X, Y):
    return anisotropic_rbf_kernel(X, Y, Lambda=Lambda_1d)


anisotropic_kernel.__name__ = "anisotropic_rbf"

spectral_matched_kernel = spectral_matched_kernel(
    gamma=gamma,
    sigma=sigma,
    dt=dt,
    m=5,
)

spectral_mismatched_kernel = spectral_mismatched_kernel(
    gamma=gamma,
    sigma=sigma,
    dt=dt,
    m=5,
    nu=1.0,
    perm=np.array([4, 2, 0, 3, 1]),
)

kernel_specs = [
    {"name": "linear", "kernel": "linear", "params": {}},
    {"name": "poly", "kernel": "poly", "params": {"gamma": 12.5, "degree": 3, "coef0": 1.0}},
    {"name": "rbf", "kernel": "rbf", "params": {"gamma": 12.5}},
    {"name": "laplacian", "kernel": "laplacian", "params": {"gamma": 12.5}},
    {"name": "anisotropic_rbf", "kernel": anisotropic_kernel, "params": {}},
    {"name": "spectral_matched", "kernel": spectral_matched_kernel, "params": {}},
    {"name": "spectral_mismatched", "kernel": spectral_mismatched_kernel, "params": {}},
]
# ============================================================
# 4. Run experiment
# ============================================================

trials_records = []
modes_records = []

for method, reduced_rank, kernel_spec in product(
    ["Principal Components (kDMD)", "Reduced Rank"],
    [False, True],
    kernel_specs,
):
    kernel_name = kernel_spec["name"]
    kernel = kernel_spec["kernel"]
    kernel_params = kernel_spec["params"]

    desc = f"{method} | {kernel_name}"
    for trial in tqdm(range(n_trials), desc=desc):
        model = KernelRidge(
            n_components=5,
            reduced_rank=reduced_rank,
            kernel=kernel,
            alpha=1e-6,
            random_state=trial,
            **kernel_params,
        )
        model.fit(data)

        n = model.kernel_X_.shape[0]
        C = model.kernel_X_ / n
        T = model.kernel_YX_ / n

        vals_hat, funcs_hat = model.eig(eval_right_on=model.X_fit_[:-1])
        sort_perm = np.flip(np.argsort(np.abs(vals_hat)))
        vals_hat = vals_hat[sort_perm]
        funcs_hat = funcs_hat[:, sort_perm]

        gap = spectral_gap(vals_hat)
        n_spur = spurious_ref(vals_hat, vals_ref, delta=0.05)

        vals_check, psi_X_val_raw = model.eig(eval_right_on=X_val)
        _, psi_Y_val_raw = model.eig(eval_right_on=Y_val)

        assert np.allclose(np.sort(np.abs(vals_check)), np.sort(np.abs(vals_hat))), (
            "eig() ordering changed between calls"
        )

        psi_X_val = psi_X_val_raw[:, sort_perm]
        psi_Y_val = psi_Y_val_raw[:, sort_perm]

        n_residual, spur_scores = spurious_residual(
            vals_hat, psi_X_val, psi_Y_val, delta=0.1, relative=True
        )

        fit_rank = model.rank_
        rho = (
            kdmd_truncation(C, fit_rank)
            if method == "Principal Components (kDMD)"
            else rrr_truncation(C, T, fit_rank)
        )

        modes = model.dynamical_modes(data)
        n_modes = modes.n_modes

        trials_records.append(
            {
                "kernel": kernel_name,
                "method": method,
                "trial": trial,
                "spurious_ref_count": int(n_spur),
                "spurious_residual_count": int(n_residual),
                "spectral_gap": float(gap),
                "rank": int(fit_rank),
            }
        )

        for j in range(n_modes):
            bias, distortion = spectral_bias(funcs_hat[:, j], C, rho)

            modes_records.append(
                {
                    "kind": "N/A",
                    "kernel": kernel_name,
                    "method": method,
                    "trial": trial,
                    "eigenfunction_id": j + 1,
                    "spectral_bias": float(bias),
                    "metric_distortion": float(distortion),
                    "truncation": float(rho),
                    "residual_spurious_score": float(spur_scores[j]),
                    "spectral_gap": float(gap),
                    "est_eig_real": float(np.real(vals_hat[j])),
                    "est_eig_imag": float(np.imag(vals_hat[j])),
                }
            )


# ============================================================
# 5. Analyse
# ============================================================

modes_df, trials_df, summary, corr_df, fig1, fig2 = analyse_spectrum(
    modes_records, trials_records, out_prefix="langevin_kernels"
)

print(modes_df)
print(trials_df)
print(summary)
print(corr_df)
